# Analysis helpers

`emis_summary()` rolls emissions up by pollutant, process, time or vehicle
variables. `emis_grid()` allocates segment emissions to polygon cells by
intersected length and conserves mass.


In [ ]:
from pathlib import Path

import geopandas as gpd
import pandas as pd
from shapely.geometry import box

from gtfs2emis_py import emis_grid, emis_summary, emission_model, transport_model

DATA = Path.cwd() / "tests/data"
tp = transport_model(DATA / "synthetic_edge_cases.zip", parallel=False)
fleet = pd.DataFrame(
    {
        "type_name_br": ["BUS_URBAN_D"],
        "model_year": [2015],
        "fuel": ["D"],
        "fleet_composition": [1.0],
    }
)
result = emission_model(tp, "ef_brazil_cetesb", fleet, ["CO", "NOx"])
emis_summary(result, by=["pollutant", "time"]).head()

In [ ]:
minx, miny, maxx, maxy = tp.table.total_bounds
grid = gpd.GeoDataFrame(geometry=[box(minx, miny, maxx, maxy)], crs=tp.table.crs)
allocated = emis_grid(result, grid)
print("emitted grams:", result.emi["value"].sum())
print("allocated grams:", allocated["value"].sum())
allocated

`segment_vars` accepts `route_id`, `shape_id`, `trip_id` or
`stop_sequence`; columns already in the emission table are used directly,
avoiding duplicate joins.
